# Part I: Conformal Regression

We consider a simple regression problem on heteroskedastic data. We want to evaluate the uncertainty associated with the prediction using various conformal prediction methods. The main objective of this first part is to get a better grasp of how Conformal Prediction works, and to *visualize* the effect of the different algorithms on the coverage rate and the size of the prediction intervals. We will code most of the algorithms from scratch, and compare our results with those obtained with help of the PUNCC library for verification purposes.

**Links**
- [PUNCC Github](https://github.com/deel-ai/puncc)
- [PUNCC Documentation](https://deel-ai.github.io/puncc/index.html)

**Warning!** We will be working on tensorflow and not on pytorch. Make sure to use the jupyter kernel associated with the tensorflow environment !

## 0. Setup

We install the puncc library using pip

In [ ]:
pip install puncc

We import some of the libraries that we will be using throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
import logging

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

warnings.filterwarnings("ignore")

## 1. Dataset

We consider a synthetic 1D heteroskedastic dataset, where the variance of the noise increases with the value of the input feature.
We generate $N$ samples as follows:

- Inputs $X$ are uniformly distributed on $[0, 20]$
- Outputs are given by $Y = (1+\epsilon)\cdot X, $

Such that $\epsilon \sim {\cal N}(\mu=0,\sigma=1)$ is standard gaussian noise.

In [ ]:
# Generate synthetic 1D heteroskedastic data

def heteroskedastic_data(n_samples):
    X = np.random.uniform(0, 20, n_samples)
    y = (1 + np.random.randn(n_samples)) * X
    X = X.reshape(-1, 1)
    return X, y

n_samples = 4_000
X, y = heteroskedastic_data(n_samples)

def plot_data(X, y):
    plt.figure(figsize=(10, 8))
    plt.scatter(X, y, alpha=0.6)
    plt.xlabel(r"$X$")
    plt.ylabel(r"$Y$")
    plt.xticks(np.arange(22, step=2))
    plt.tight_layout()
    plt.grid(False)
    plt.show()

plot_data(X, y)

We split the data into a training set and a test set by using the `train_test_split` function of the `scikit-learn` library. Split the data randomly and leaving out 25% of the data for the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

## 2. Split Conformal Regression

In order to perform the *split conformal regression* algorithm, we need to split our training data into a *proper training set* (which we will store in the variables `X_fit` and `y_fit`) and a *calibration set*.

**Exercise.** Further split the training data randomly by leaving out 50% of the training data for the calibration set.

In [ ]:
# TODO: Split the data

In [ ]:
# %load solutions/split/split_calib.py

Next we train a prediction model using the `LinearRegression` class in the `scikit-learn` library.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_fit, y_fit)

The `plot_model` function below plots the synthetic data along with the model predictions. We use the function `plot_model` to plot the test data along with the model predictions.

In [ ]:
def plot_model(model, X, y):
    plt.figure(figsize=(10, 8))
    plt.scatter(X, y, alpha=0.6)
    plt.plot(X, model.predict(X), color="red", linewidth=3)
    plt.xlabel(r"$X$")
    plt.ylabel(r"$Y$")
    plt.xticks(np.arange(22, step=2))
    plt.tight_layout()
    plt.grid(False)
    plt.show()

plot_model(linear_model, X_test, y_test)

Now that we have a trained model, we wish to *conformalize* it. The simplest algorithm to conformalize a model is the *split conformal* algorithm.

In this section, we are using the split conformal algorithm for the regression task, but as we have seen in the lectures, there are many other conformalization algorithms that rely on splitting the data into a train/fit and a calibration set. Therefore, we will code a generic class called `SplitConformal` that we will be able to use later for other conformal prediction algorithms of the *split* type.

In order to define our general `SplitConformal` class, we will rely on the following information:
the algorithms using the `SplitConformal` class are different between each other only in two ways:
- the way the *nonconformity scores* are computed,
- the way the *prediction sets* are constructed.

Therefore, we will build the `SplitConformal` class so that it can work with any choice of *nonconformity score function* and *prediction set contruction function*.


**Exercise.** Code the `SplitConformal` class by implementing the following:
1. Attributes:
  - `score_fn`: the nonconformity score function to be used (a function that computes scores between ground truth values and model predictions).
  - `predset_fn`: the prediction set construction function to be used (a function that builds prediction sets from model predictions and quantile values).
  - `scores`: attribute to store the array of calibration nonconformity scores.
  - `quantile`: attribute to store the value of the quantile of order $1-\alpha$ (plus correction).

2. Methods:
  - `__init__`: takes as input the choice of nonconformity score function and prediction set construction function.
  - `compute_scores`: takes as input a numpy array containing ground truth $y$-values and predicted $y$-values, computes the nonconformity scores, and stores them into the attribute `scores`.
  - `compute_quantile`: takes as input the nominal error rate $\alpha$ and computes the quantile of level $1-\alpha$ (taking into account the usual finite-sample correction characteristic of conformal prediction algorithms) on the array of scores saved in the `scores` attribute. **Warning!** You may use the `quantile` function in the `numpy` library, however, make sure you choose the *right* method to compute the quantile values, otherwise the probabilistic guarantee of conformal prediction will no longer be true. (**This is an important point, try and understand the different methods and choose the correct one yourself before checking the solutions!!!**)
  - `predict`: takes as input an array of model predictions and outputs prediction sets (in whatever format the `predset_fn` outputs prediction sets).

In [ ]:
class SplitConformal():
    # TODO: Complete the class

In [ ]:
# %load solutions/split/split_conformal.py

We are dealing with a univariate regression problem. Thus, a natural choice for our nonconformity score is to use the absolute difference between the ground truth and the prediction. The corresponding prediction set is an interval, which is obtained by adding and substracting the empirical quantile to the model prediction :
- $S(y,\hat{y}) = |y-\hat{y}|$
- $C_{\alpha}(x) = [\hat{f}(x)-\hat{q}_{1-\alpha}, \hat{f}(x)+\hat{q}_{1-\alpha}]$

**Exercise.** Code the `abs_difference` and `additive_interval` functions to be used as the arguments of the `SplitConformal` class at initialization.

In [ ]:
# TODO: code the abs_difference function

# TODO: code the additive_interval function

In [ ]:
# %load solutions/split/absdiff_addint.py

**Exercise.** Use the `SplitConformal` class to conformalize the Linear Regression model with a nominal error rate of 0.1.

In [ ]:
# TODO: Conformalize

alpha = ...
y_pred_calib = ...
splitcr = ...
# TODO: compute the nonconformity scores on the calibration set
# TODO: compute the quantile of the nonconformity scores

y_pred_test = ...
y_lower, y_upper = ... 

In [ ]:
# %load solutions/split/conformalize_regressor.py

The function `plot_conformalized_data` below plots the test dataset along with the model predictions and the prediction intervals.

In [ ]:
def plot_conformalized_data(X, y, y_pred, y_lower, y_upper):
    plt.figure(figsize=(10, 8))
    sort_indices = np.argsort(X.flatten())

    plt.scatter(X, y, alpha=0.6)
    plt.plot(X, y_pred, color="red", linewidth=3)
    plt.fill_between(
        X[sort_indices].flatten(),
        y_lower[sort_indices],
        y_upper[sort_indices],
        alpha=0.3,
        color="green",
    )
    plt.xlabel(r"$X$")
    plt.ylabel(r"$Y$")
    plt.xticks(np.arange(22, step=2))
    plt.tight_layout()
    plt.grid(False)
    plt.show()

plot_conformalized_data(X_test, y_test, y_pred_test, y_lower, y_upper)

**Exercise.** Write a function called `evaluate_conformal_regression` that takes as input an array of ground-truth $y$-values along with an array of the lower limits and an array of the upper limits of the corresponding prediction intervals. It then computes and outputs the following two metrics:
1. `coverage`: The average number of intervals that contain the ground-truth values.
2. `avg_length`: The average length of the intervals.

**Questions.**
1. What value should we expect for the `coverage` metric?
2. What kind of values do we desire for the `avg_length` metric?


In [ ]:
# TODO: implement the evaluate_conformal_regression function

In [ ]:
# %load solutions/split/eval_conf_regressor.py

**Exercise.** Evaluate the conformalized model with the help of the `evaluate_conformal_regression` function and display the results.

In [ ]:
# TODO: evaluate the conformal prediction regressor

The evaluation seems to match the desired results. 

**Remark.** Note however (from the plot) how the interval length is constant, and does not match the heteroskedasticity of the data. Indeed, the theoretical guarantee in the CP theorem is true for the quantity $P(Y \in C_{\alpha}(X))$ averaged over $X$, but is not true in general for $P(Y\in C_{\alpha}(X)| X=x)$!

## 4. Conformal Quantile Regression
We now turn to the problem of the *constant* prediction intervals. As we can see in the plots above, the coverage rate of $1-\alpha$ is obtained by over-covering in the low-variance regions and under-covering in the high variance regions. We consider the *Conformalized Quantile Regression (CQR)* algorithm, the purpose of which is to generate prediction sets that are more adapted to the heteroskedasticity of the data. CQR extends traditional quantile regression by incorporating conformal prediction techniques, allowing us to construct predictive intervals with state-of-the-art performance and guaranteed coverage (under data exchangeability).
 
We start by training lower and upper quantile models for a nominal eror rate $\alpha$ of 0.1, using the `GradientBoostingRegressor` model with 10 estimators from the `sklearn.ensemble` module.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Lower quantile regressor
lower_quantile_model = GradientBoostingRegressor(
    loss="quantile", alpha=alpha / 2, n_estimators=10
)
# Upper quantile regressor
upper_quantile_model = GradientBoostingRegressor(
    loss="quantile", alpha=1 - alpha / 2, n_estimators=10
)

# Train the models
lower_quantile_model.fit(X_fit, y_fit)
upper_quantile_model.fit(X_fit, y_fit)

**Exercise.** Complete the `plot_quantile_model` function below and use it to plot the test examples along with the lower and upper quantile model predictions.

In [ ]:
def plot_quantile_models(X, y, lower_quantile_model, upper_quantile_model):
    plt.figure(figsize=(10, 8))
    sort_indices = np.argsort(X.flatten())
    X_sorted = X[sort_indices]
    y_sorted = y[sort_indices]
    plt.scatter(X, y, alpha=0.6)
    plt.plot(X_sorted, lower_quantile_model.predict(X_sorted), color="red", linewidth=3)
    plt.plot(X_sorted, upper_quantile_model.predict(X_sorted), color="red", linewidth=3)
    plt.xlabel(r"$X$")
    plt.ylabel(r"$Y$")
    plt.xticks(np.arange(22, step=2))
    plt.tight_layout()
    plt.grid(False)
    plt.show()


plot_quantile_models(X_test, y_test, lower_quantile_model, upper_quantile_model)

**Exercise.** Code the functions `cqr_score` and `cqr_set` so that we can use them along with the class `SplitConformal` above in order to implement the CQR algorithm. In order to keep a structure compatible with the `SplitConformal` class above, the `y_pred` input to the `cqr_score` and `cqr_set` functions is an array of length 2, where the first element contains the predictions of the lower quantile model and the second one the predictions of the upper quantile model.

In [ ]:
# TODO: implement the functions

In [ ]:
# %load solutions/cqr/cqr_functions.py

**Exercise.** Conformalize the quantile regressors using the CQR algorithm and the calibration dataset.

In [ ]:
# TODO: Conformalize according to the CQR algorithm
y_pred_calib_lower = # TODO: predict the upper quantiles of the calibration data
y_pred_calib_upper = # TODO: predict the lower quantiles of the calibration data
y_pred_calib = np.stack([y_pred_calib_lower, y_pred_calib_upper])

alpha = # TODO: set the value of alpha

splitcr = # TODO: instantiate the split conformal class

# TODO: conformalize the quantile regression model

# TODO: compute the prediction intervals on the test data

In [ ]:
# %load solutions/cqr/cqr_calib.py

**Exercise.** Evaluate and visualize the results of the CQR conformalized model.

In [ ]:
# TODO: evaluate the results and print the values of the coverage and interval length

def plot_cqr_conformalized_data(X, y, y_pred_lower, y_pred_upper, y_lower, y_upper):
    plt.figure(figsize=(10, 8))
    sort_indices = np.argsort(X.flatten())
    X_sorted = X[sort_indices]
    y_pred_lower_sorted = y_pred_lower[sort_indices]
    y_pred_upper_sorted = y_pred_upper[sort_indices]
    y_lower_sorted = y_lower[sort_indices]
    y_upper_sorted = y_upper[sort_indices]

    plt.scatter(X, y, alpha=0.6)
    plt.plot(X_sorted, y_pred_lower_sorted, color="red", linewidth=3)
    plt.plot(X_sorted, y_pred_upper_sorted, color="red", linewidth=3)
    plt.fill_between(
        X_sorted.flatten(),
        y_lower_sorted,
        y_upper_sorted,
        alpha=0.3,
        color="green",
    )
    plt.xlabel(r"$X$")
    plt.ylabel(r"$Y$")
    plt.xticks(np.arange(22, step=2))
    plt.tight_layout()
    plt.grid(False)
    plt.show()

plot_cqr_conformalized_data(X_test, y_test, y_pred_test_lower, y_pred_test_upper, y_lower, y_upper)

In [ ]:
# %load solutions/cqr/cqr_eval.py

**Questions.**
- What do you observe?
- Is this the expected behavior? Why?
- How does the average interval width compare with the previous methods?

# Part II: Conformal Classification

The objective of this section is to train a small neural network on the MNIST dataset and apply the Conformal Classification algorithms seen during the lecture.

 ## 1. Dataset
We start with the following steps :
1. Import the MNIST dataset from keras.
2. Split the training set into a proper training set, which we call the `fit` dataset, and a calibration set. Use the first 50_000 training points for the fit dataset and the remaining ones for the calibration dataset.
3. Convert the labels to categorical, and save them into new arrays.

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load MNIST Database
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocessing: reshaping and standardization
X_train = X_train.reshape((len(X_train), 28, 28))
X_train = X_train.astype('float32') / 255
X_test = X_test.reshape((len(X_test), 28 , 28))
X_test = X_test.astype('float32') / 255

# Split fit and calib datasets
X_fit, X_calib  = X_train[:50000], X_train[50000:]
y_fit, y_calib  = y_train[:50000], y_train[50000:]

# One hot encoding of classes
y_fit_cat = to_categorical(y_fit)
y_calib_cat = to_categorical(y_calib)
y_test_cat = to_categorical(y_test)

## 2. Model

We define a simple convolutional neural network having the following sequential architecture:
- A convolution with kernel size 3 and 16 channels.
- A ReLU activation.
- A max pooling layer with kernel size 2.
- A convolution with kernel size 3 and 32 channels.
- A reLU activation.
- A max pooling layer with kernel size 2.
- A Fully connected layer with 10 neurons.
- A softmax activation.

In [ ]:
from tensorflow import random
from tensorflow import keras
from tensorflow.keras import layers

random.set_seed(0)
keras.utils.set_random_seed(0)

# Classification model: convnet composed of two convolution/pooling layers
# and a dense output layer
nn_model = keras.Sequential(
   [
      keras.Input(shape=(28, 28, 1)),
      layers.Conv2D(16, kernel_size=(3, 3), activation="relu"),
      layers.MaxPooling2D(pool_size=(2, 2)),
      layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
      layers.MaxPooling2D(pool_size=(2, 2)),
      layers.Flatten(),
      layers.Dense(10, activation="softmax"),
   ]
)

## 3. Training

We train the model using the Adam optimizer, and the categorical cross-entropy loss. Plot the training and validation accuracy while training, with 10% of the fit data left out for the validation set.

We train the model for 2 epochs with a batch size of 256.

In [ ]:
# Train the model for two epochs
nn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
nn_model.fit(X_fit, y_fit_cat, epochs=2, batch_size=256, validation_split=0.1, verbose=1)

## 4. Least Ambiguous Set-Valued Classifiers (LAC)
We now implement the LAC algorithm seen during the lecture.

**Exercise.** Define a `lac_score` and a `lac_set` function in order to be used with the `SplitConformal` class above.

In [ ]:
# TODO: define both functions

In [ ]:
# %load solutions/classif/lac_funcs.py

**Exercise.** Conformalize the classification model.

In [ ]:
y_pred_calib = nn_model.predict(X_calib)
y_pred_test = nn_model.predict(X_test)

In [ ]:
# TODO: Conformalize the classifier using the calibration dataset and a nominal error rate of 0.1

In [ ]:
# %load solutions/classif/lac_calib.py

**Exercise.** Evaluate the results by computing the average coverage and average size of the prediction sets on the test set.

In [ ]:
def eval_conformal_classifier(y_true, y_predset):
    # TODO: implement the function to compute both metrics (coverage and set size)

# TODO: Evaluate the conformalized model.

In [ ]:
# %load solutions/classif/lac_eval.py

**Exercise.** Plot a random image along with the prediction set.

In [ ]:
sample = 18

plt.imshow(X_test[sample].reshape((28,28)))
_ = plt.title(f"Point prediction: {np.argmax(y_pred_test[sample])} \n Prediction set: {y_predset_test[sample]}")
_ = plt.xticks([])
_ = plt.yticks([])

**Questions.**
1. How come the average size of the prediction sets is smaller than 1?
2. Some of the prediction sets are empty, they contain no labels. Why? Do you think that this is an Ok behavior?